# USL Championship — Player Data

Outfield player and goalkeeper metrics using the ASA public API.

In [ ]:
import sys
from typing import Final

import pandas as pd
from itscalledsoccer.client import AmericanSoccerAnalysis

sys.path.insert(0, "../scripts")
from utils import flatten_goals_added, resolve_team

pd.options.display.max_columns = 999
pd.set_option("expand_frame_repr", False)

asa_client = AmericanSoccerAnalysis()

LEAGUE: Final[str] = "uslc"
GK_MIN_MINUTES: Final[int] = 100  # minimum minutes to qualify a GK for rate stat analysis


## Data Fetch

In [ ]:
players_merge = asa_client.get_players(leagues=[LEAGUE])[
    ["player_id", "player_name", "birth_date", "nationality", "height_ft", "height_in"]
].copy()

players_merge["height_ft"] = players_merge["height_ft"].astype("Int64")
players_merge["height_in"] = players_merge["height_in"].astype("Int64")

# Build HEIGHT string only where both components are present;
# Int64 NA values stringify to "<NA>" without this guard.
_has_ht = players_merge["height_ft"].notna() & players_merge["height_in"].notna()
players_merge["HEIGHT"] = (
    players_merge["height_ft"].astype(str)
    + "' "
    + players_merge["height_in"].astype(str)
    + '"'
).where(_has_ht)
players_merge = players_merge.drop(columns=["height_ft", "height_in"])

In [ ]:
players_xg = asa_client.get_player_xgoals(
    leagues=[LEAGUE], split_by_seasons=True, stage_name="Regular Season"
)
players_xp = asa_client.get_player_xpass(
    leagues=[LEAGUE], split_by_seasons=True, stage_name="Regular Season"
)
players_ga_raw = asa_client.get_player_goals_added(
    leagues=[LEAGUE], split_by_seasons=True, stage_name="Regular Season"
)
gk_xg = asa_client.get_goalkeeper_xgoals(
    leagues=[LEAGUE], split_by_seasons=True, stage_name="Regular Season"
)
gk_ga_raw = asa_client.get_goalkeeper_goals_added(
    leagues=[LEAGUE], split_by_seasons=True, stage_name="Regular Season"
)

## Data Cleaning

Build team lookup, resolve `team_id` to abbreviation, and prepare xPass for joining.

In [ ]:
teams = asa_client.get_teams(leagues=[LEAGUE])
team_map: dict[str, str] = dict(zip(teams["team_id"], teams["team_abbreviation"]))


players_xg["team_name"] = players_xg["team_id"].apply(resolve_team, team_map=team_map)
players_xg = players_xg.drop(columns=["team_id"])

gk_xg["team_name"] = gk_xg["team_id"].apply(resolve_team, team_map=team_map)
gk_xg = gk_xg.drop(columns=["team_id"])

# Drop columns from xPass that would create merge conflicts.
# - team_id: already resolved to team_name in xGoals
# - general_position: present in xGoals; xPass version is redundant
# - minutes_played: present in xGoals; xPass version is redundant
# count_games is unique to xPass (xGoals does not return it) and is kept.
players_xp = players_xp.drop(
    columns=["team_id", "general_position", "minutes_played"],
    errors="ignore",
)
players_xp["season_name"] = players_xp["season_name"].astype("Int64")

# Flatten Goals Added frames into wide per-action columns.
players_ga = flatten_goals_added(players_ga_raw)
players_ga["season_name"] = players_ga["season_name"].astype("Int64")
gk_ga = flatten_goals_added(gk_ga_raw)
gk_ga["season_name"] = gk_ga["season_name"].astype("Int64")

## Outfield Players

In [ ]:
players = players_merge.merge(players_xg, on="player_id", how="left")
players = players[players["season_name"].notna()]

players["season_name"] = players["season_name"].astype("Int64")
players["minutes_played"] = players["minutes_played"].astype("Int64")

players["MinPerShot"] = (players["minutes_played"] / players["shots"]).round(1)
players["MinPerSOT"] = (players["minutes_played"] / players["shots_on_target"]).round(1)
players["MinPerG"] = (players["minutes_played"] / players["goals"]).round(1)
players["MinPerXG"] = (players["minutes_played"] / players["xgoals"]).round(2)
players["ShtPerSOT"] = (players["shots"] / players["shots_on_target"]).round(2)
players["ShtPerG"] = (players["shots"] / players["goals"]).round(1)
players["ShtPerXG"] = (players["shots"] / players["xgoals"]).round(2)
players["GPerShot"] = (players["goals"] / players["shots"]).round(2)
players["xGPerShot"] = (players["xgoals"] / players["shots"]).round(3)
players["MinPerAssist"] = (players["minutes_played"] / players["primary_assists"]).round(2)
players["MinPerXA"] = (players["minutes_played"] / players["xassists"]).round(2)

# Replace inf values that arise when a denominator is zero (e.g. 0 shots, 0 goals).
players = players.replace([float("inf"), float("-inf")], pd.NA)

players["xgoals"] = players["xgoals"].round(2)

In [ ]:
players = players.merge(
    players_xp,
    on=["player_id", "season_name"],
    how="left",
)
players["PassAttemptsPerGame"] = (players["attempted_passes"] / players["count_games"]).round(2)

In [ ]:
# Merge in flattened Goals Added (per-action + totals).
_ga_join = ["player_id", "season_name"]
_ga_extra = [c for c in players_ga.columns if c not in players.columns]
players = players.merge(
    players_ga[_ga_join + _ga_extra], on=_ga_join, how="left"
)

In [ ]:
# Per-90 normalization — the standard rate basis in football analytics.
_per90 = 90 / players["minutes_played"]
players["G_p90"] = (players["goals"] * _per90).round(2)
players["xG_p90"] = (players["xgoals"] * _per90).round(2)
players["A_p90"] = (players["primary_assists"] * _per90).round(2)
players["xA_p90"] = (players["xassists"] * _per90).round(2)
players["GA_p90"] = (players["goals_plus_primary_assists"] * _per90).round(2)
players["xGA_p90"] = (players["xgoals_plus_xassists"] * _per90).round(2)
players["shots_p90"] = (players["shots"] * _per90).round(2)
players["sot_p90"] = (players["shots_on_target"] * _per90).round(2)
players["key_passes_p90"] = (players["key_passes"] * _per90).round(2)
players["ga_raw_p90"] = (players["ga_raw_total"] * _per90).round(3)

In [ ]:
# Player attributes: age at season, multi-team flag, position group.
_birth = pd.to_datetime(players["birth_date"], errors="coerce")
players["age"] = (
    players["season_name"].astype("Int64") - _birth.dt.year.astype("Int64")
).astype("Int64")

players["is_multi_team"] = players["team_name"].str.contains("/", na=False)

_position_group_map: dict[str, str] = {
    "GK": "Goalkeeper",
    "CB": "Defender", "FB": "Defender",
    "DM": "Midfielder", "CM": "Midfielder", "AM": "Midfielder",
    "W": "Attacker", "ST": "Attacker",
}
players["position_group"] = players["general_position"].map(_position_group_map)

In [ ]:
# Within-position-season percentile ranks for key per-90 + efficiency stats.
# Higher = better; computed within (general_position, season_name) cohort.
_rank_cols = [
    "G_p90", "xG_p90", "A_p90", "xA_p90", "GA_p90", "xGA_p90",
    "shots_p90", "key_passes_p90", "ga_raw_p90",
    "share_team_touches", "pass_completion_percentage",
    "passes_completed_over_expected_p100",
]
for _c in _rank_cols:
    players[f"{_c}_pct"] = (
        players.groupby(["general_position", "season_name"])[_c]
        .rank(pct=True)
        .round(3)
    )

# Final inf cleanup after all derived columns.
players = players.replace([float("inf"), float("-inf")], pd.NA)

In [ ]:
_col_order = [
    # Identity / bio
    "player_id", "player_name", "birth_date", "age", "nationality", "HEIGHT",
    # Context
    "team_name", "is_multi_team", "season_name",
    "general_position", "position_group",
    "count_games", "minutes_played",
    # Shooting — volume
    "shots", "shots_on_target", "goals", "xgoals", "goals_minus_xgoals", "xplace",
    # Shooting — rates
    "GPerShot", "xGPerShot",
    "MinPerShot", "MinPerSOT", "MinPerG", "MinPerXG",
    "ShtPerSOT", "ShtPerG", "ShtPerXG",
    # Assists
    "key_passes", "primary_assists", "xassists", "primary_assists_minus_xassists",
    "MinPerAssist", "MinPerXA",
    # Combined
    "goals_plus_primary_assists", "xgoals_plus_xassists",
    "points_added", "xpoints_added",
    # Passing
    "attempted_passes", "PassAttemptsPerGame",
    "pass_completion_percentage", "xpass_completion_percentage",
    "passes_completed_over_expected", "passes_completed_over_expected_p100",
    "avg_distance_yds", "avg_vertical_distance_yds", "share_team_touches",
    # Goals Added — totals
    "ga_raw_total", "ga_aboveavg_total",
    # Goals Added — per action (raw)
    "ga_raw_dribbling", "ga_raw_fouling", "ga_raw_interrupting",
    "ga_raw_passing", "ga_raw_receiving", "ga_raw_shooting",
    # Goals Added — per action (above average)
    "ga_aboveavg_dribbling", "ga_aboveavg_fouling", "ga_aboveavg_interrupting",
    "ga_aboveavg_passing", "ga_aboveavg_receiving", "ga_aboveavg_shooting",
    # Goals Added — action counts
    "ga_actions_dribbling", "ga_actions_fouling", "ga_actions_interrupting",
    "ga_actions_passing", "ga_actions_receiving", "ga_actions_shooting",
    # Per-90 rates
    "G_p90", "xG_p90", "A_p90", "xA_p90", "GA_p90", "xGA_p90",
    "shots_p90", "sot_p90", "key_passes_p90", "ga_raw_p90",
    # Within-position-season percentile ranks
    "G_p90_pct", "xG_p90_pct", "A_p90_pct", "xA_p90_pct",
    "GA_p90_pct", "xGA_p90_pct",
    "shots_p90_pct", "key_passes_p90_pct", "ga_raw_p90_pct",
    "share_team_touches_pct", "pass_completion_percentage_pct",
    "passes_completed_over_expected_p100_pct",
]
players = players[_col_order]
players = players.sort_values(["player_name", "season_name"]).reset_index(drop=True)
display(players.head())

## Goalkeepers

In [ ]:
gk_players = players_merge.merge(gk_xg, on="player_id", how="left")
gk_players = gk_players[gk_players["season_name"].notna()]
gk_players = gk_players[gk_players["shots_faced"] > 0]
gk_players = gk_players[gk_players["minutes_played"] >= GK_MIN_MINUTES]  # excludes backup/emergency appearances

gk_players["season_name"] = gk_players["season_name"].astype("Int64")
gk_players["minutes_played"] = gk_players["minutes_played"].astype("Int64")

gk_players["MINUTES PER SHOT"] = (
    gk_players["minutes_played"] / gk_players["shots_faced"]
).round(1)
gk_players["MINUTES PER GOAL"] = (
    gk_players["minutes_played"] / gk_players["goals_conceded"]
).round(1)
gk_players["MINUTES PER SAVE"] = (
    gk_players["minutes_played"] / gk_players["saves"]
).round(1)
gk_players["MINUTES PER XG"] = (
    gk_players["minutes_played"] / gk_players["xgoals_gk_faced"]
).round(1)
gk_players["SHOTS PER GOAL"] = (
    gk_players["shots_faced"] / gk_players["goals_conceded"]
).round(2)
gk_players["SAVES PER GOAL"] = (
    gk_players["saves"] / gk_players["goals_conceded"]
).round(1)
gk_players["XG PER GOAL"] = (
    gk_players["xgoals_gk_faced"] / gk_players["goals_conceded"]
).round(2)
gk_players["XG PER SHOT"] = (
    gk_players["xgoals_gk_faced"] / gk_players["shots_faced"]
).round(2)

# GKs with 0 goals_conceded or 0 saves produce inf in several rate columns.
gk_players = gk_players.replace([float("inf"), float("-inf")], pd.NA)

In [ ]:
# Merge in flattened GK Goals Added (per-action + totals).
_gk_ga_join = ["player_id", "season_name"]
_gk_ga_extra = [c for c in gk_ga.columns if c not in gk_players.columns]
gk_players = gk_players.merge(
    gk_ga[_gk_ga_join + _gk_ga_extra], on=_gk_ga_join, how="left"
)

In [ ]:
# GK derived metrics: save %, count_games estimate, per-90 rates, attributes, and percentile ranks.
gk_players["save_pct"] = (
    gk_players["saves"] / gk_players["shots_faced"]
).round(3)

gk_players["count_games_est"] = (gk_players["minutes_played"] / 90).round(1)

_per90 = 90 / gk_players["minutes_played"]
gk_players["goals_conceded_p90"] = (gk_players["goals_conceded"] * _per90).round(2)
gk_players["saves_p90"] = (gk_players["saves"] * _per90).round(2)
gk_players["shots_faced_p90"] = (gk_players["shots_faced"] * _per90).round(2)
gk_players["xgoals_faced_p90"] = (gk_players["xgoals_gk_faced"] * _per90).round(2)
gk_players["ga_raw_p90"] = (gk_players["ga_raw_total"] * _per90).round(3)

# Attributes mirroring outfield frame.
_gk_birth = pd.to_datetime(gk_players["birth_date"], errors="coerce")
gk_players["age"] = (
    gk_players["season_name"].astype("Int64") - _gk_birth.dt.year.astype("Int64")
).astype("Int64")

gk_players["is_multi_team"] = gk_players["team_name"].str.contains("/", na=False)

# Within-season percentile ranks. All GKs share one position so no position grouping needed.
# Higher is better for save metrics; lower is better for conceded/xG metrics.
_gk_rank_high = ["save_pct", "saves_p90", "ga_raw_p90"]
_gk_rank_low = ["goals_conceded_p90", "goals_minus_xgoals_gk", "goals_divided_by_xgoals_gk"]
for _c in _gk_rank_high:
    gk_players[f"{_c}_pct"] = (
        gk_players.groupby("season_name")[_c].rank(pct=True).round(3)
    )
for _c in _gk_rank_low:
    gk_players[f"{_c}_pct"] = (
        gk_players.groupby("season_name")[_c].rank(pct=True, ascending=False).round(3)
    )

gk_players = gk_players.replace([float("inf"), float("-inf")], pd.NA)

In [ ]:
_col_order = [
    # Identity / bio
    "player_id", "player_name", "birth_date", "age", "nationality", "HEIGHT",
    # Context
    "team_name", "is_multi_team", "season_name", "minutes_played", "count_games_est",
    # Volume
    "shots_faced", "goals_conceded", "saves", "share_headed_shots",
    # xG
    "xgoals_gk_faced", "goals_minus_xgoals_gk", "goals_divided_by_xgoals_gk",
    # Rate stats
    "save_pct",
    "MINUTES PER SHOT", "MINUTES PER GOAL", "MINUTES PER SAVE", "MINUTES PER XG",
    "SHOTS PER GOAL", "SAVES PER GOAL", "XG PER GOAL", "XG PER SHOT",
    # Per-90
    "goals_conceded_p90", "saves_p90", "shots_faced_p90",
    "xgoals_faced_p90", "ga_raw_p90",
    # Within-season percentile ranks
    "save_pct_pct", "saves_p90_pct", "ga_raw_p90_pct",
    "goals_conceded_p90_pct", "goals_minus_xgoals_gk_pct", "goals_divided_by_xgoals_gk_pct",
    # Goals Added — totals
    "ga_raw_total", "ga_aboveavg_total",
    # Goals Added — per action (raw)
    "ga_raw_claiming", "ga_raw_fielding", "ga_raw_handling",
    "ga_raw_passing", "ga_raw_shotstopping", "ga_raw_sweeping",
    # Goals Added — per action (above average)
    "ga_aboveavg_claiming", "ga_aboveavg_fielding", "ga_aboveavg_handling",
    "ga_aboveavg_passing", "ga_aboveavg_shotstopping", "ga_aboveavg_sweeping",
    # Goals Added — action counts
    "ga_actions_claiming", "ga_actions_fielding", "ga_actions_handling",
    "ga_actions_passing", "ga_actions_shotstopping", "ga_actions_sweeping",
]
gk_players = gk_players[_col_order]
gk_players = gk_players.sort_values(["player_name", "season_name"]).reset_index(drop=True)
display(gk_players.head())

In [ ]:
players.to_parquet("../data/players.parquet", index=False)
gk_players.to_parquet("../data/gk_players.parquet", index=False)